# What is Binarization?

- Binarization converts continuous features into binary (0/1) values by thresholding.
- Importance: simplifies models, reduces noise, highlights presence/absence signals, and can improve model robustness when magnitude is less informative than occurrence.
- Use cases: converting 'has-family' vs 'alone', 'high fare' vs 'low fare', or 'young' vs 'old' where membership matters more than exact value.
- Best practice: compute thresholds on training data only, validate via cross-validation, and try several thresholds if unsure.

# **Import Libraries**

In [37]:
import numpy as np
import pandas as pd

In [38]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [39]:
df = pd.read_csv('train.csv')[['Age','Fare','SibSp','Parch','Survived']]
df.dropna(inplace=True)
df.head()

,Age,Fare,SibSp,Parch,Survived
0,22.0,7.2500,1,0,0
1,38.0,71.2833,1,0,1
2,26.0,7.9250,0,0,1
3,35.0,53.1000,1,0,1
4,35.0,8.0500,0,0,0


In [40]:
df['family'] = df['SibSp'] + df['Parch']
df.head()

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [41]:
df.drop(columns=['SibSp','Parch'],inplace=True)
df.head()

,Age,Fare,Survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [42]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [43]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [44]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


# **Applying Binarization**


- Binarize `family` (already done) but choose a threshold that captures 'has family' vs 'alone'.
- Binarize `Age` and `Fare` using medians to capture 'young/old' and 'low/high fare' instead of raw values.
- Ensure transformations are computed on training data only and applied to test data.
- Use cross-validation to confirm improvements and try alternate thresholds.

In [45]:

from sklearn.preprocessing import Binarizer

In [46]:
# compute thresholds on training data and build binarizers
med_age = X_train['Age'].median()
med_fare = X_train['Fare'].median()
trf = ColumnTransformer([
    ('bin_family', Binarizer(threshold=0.5, copy=False), ['family']),
    ('bin_age', Binarizer(threshold=med_age, copy=False), ['Age']),
    ('bin_fare', Binarizer(threshold=med_fare, copy=False), ['Fare'])
], remainder='passthrough')

In [47]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [48]:
pd.DataFrame(X_train_trf,columns=['family','Age','Fare'])

,family,Age,Fare
0,1.0,1.0,1.0
1,1.0,0.0,0.0
2,1.0,1.0,1.0
3,0.0,1.0,0.0
4,0.0,0.0,0.0
...,...,...,...
566,1.0,1.0,1.0
567,0.0,0.0,0.0
568,0.0,1.0,1.0
569,1.0,1.0,1.0


In [49]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

0.6503496503496503

In [50]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

np.float64(0.6275821596244132)

# **Evaluating Alternatives**


In [52]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Binarizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# reload raw data and impute missing values (keep more rows than dropna)
df_raw = pd.read_csv('train.csv')[['Age','Fare','SibSp','Parch','Survived']]
X_full = df_raw.drop(columns=['Survived'])
y_full = df_raw['Survived']
imp = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imp.fit_transform(X_full), columns=X_full.columns)
# recreate family feature
X_imp['family'] = X_imp['SibSp'] + X_imp['Parch']
X_imp.drop(columns=['SibSp','Parch'], inplace=True)
X_cont = X_imp[['Age','Fare','family']]

# (A) DecisionTree on continuous features
scores_dt_cont = cross_val_score(DecisionTreeClassifier(random_state=42), X_cont, y_full, cv=10, scoring='accuracy')
# (B) DecisionTree on binarized features (median thresholds)
med_age = X_cont['Age'].median()
med_fare = X_cont['Fare'].median()
trf_bin = ColumnTransformer([
    ('bin_family', Binarizer(threshold=0.5), ['family']),
    ('bin_age', Binarizer(threshold=med_age), ['Age']),
    ('bin_fare', Binarizer(threshold=med_fare), ['Fare'])
], remainder='passthrough')
pipe_dt_bin = Pipeline([('trf', trf_bin), ('clf', DecisionTreeClassifier(random_state=42))])
scores_dt_bin = cross_val_score(pipe_dt_bin, X_cont, y_full, cv=10, scoring='accuracy')
# (C) RandomForest on continuous features
scores_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_cont, y_full, cv=10, scoring='accuracy')

print('DecisionTree (continuous) mean CV accuracy:', scores_dt_cont.mean())
print('DecisionTree (binarized) mean CV accuracy:', scores_dt_bin.mean())
print('RandomForest (continuous) mean CV accuracy:', scores_rf.mean())



DecisionTree (continuous) mean CV accuracy: 0.6464794007490637
DecisionTree (binarized) mean CV accuracy: 0.6353807740324594
RandomForest (continuous) mean CV accuracy: 0.6902996254681648
